In [1]:
# Instalar PyTorch con CUDA (primero, para evitar que sentence-transformers instale CPU)
%pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Core libs
%pip install pandas==2.2.2 matplotlib==3.9.0 \
datasets==2.20.0 pyarrow==15.0.2

  Using cached pandas-2.2.2.tar.gz (4.4 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [12 lines of output]
      + meson setup C:\Users\Usuario\AppData\Local\Temp\pip-install-lm1bg080\pandas_f7fd4968ed1f4183879d87e42d66a8cf C:\Users\Usuario\AppData\Local\Temp\pip-install-lm1bg080\pandas_f7fd4968ed1f4183879d87e42d66a8cf\.mesonpy-vzdgqw3m\build -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --vsenv --native-file=C:\Users\Usuario\AppData\Local\Temp\pip-install-lm1bg080\pandas_f7fd4968ed1f4183879d87e42d66a8cf\.mesonpy-vzdgqw3m\build\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.1
      Source dir: C:\Users\Usuario\AppData\Local\Temp\pip-install-lm1bg080\pandas_f7fd4968ed1f4183879d87e42d66a8cf
      Build dir: C:\Users\Usuario\AppData\Local\Temp\pip-install-lm1bg080\pandas_f7fd4968ed1f4183879d87e42d66a8cf\.mesonpy-vzdgqw3m\build
      Build type: native build
      Project name: pandas
      Project version: 

In [ ]:
import torch
import gc
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from datasets import load_dataset
from datasets import Dataset

import torch.nn.functional as F
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)
from transformers import AutoConfig


In [4]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [5]:
dataset = load_dataset("imdb")

In [6]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_MODEL_LENGTH = 2048
MAX_LENGTH = 2048

In [ ]:
def build_prompt(review):

    prompt = f"""Review: {review}

Sentiment:"""

    return prompt


def build_full_text(review, label):

    sentiment = (
        "positive"
        if label == 1
        else "negative"
    )

    full_text = f"""Review: {review}

Sentiment: {sentiment}"""

    return full_text


def preprocess_dataset(dataset, tokenizer):

    input_ids_list = []
    attention_masks_list = []
    labels_list = []
    true_labels_list = []
    texts_list = []

    for example in dataset:

        review = example["text"]
        label  = example["label"]

        full_text  = build_full_text(review, label)
        prompt_ids = tokenizer(
            build_prompt(review), add_special_tokens=False
        )["input_ids"]
        prompt_len = len(prompt_ids)

        full = tokenizer(full_text, add_special_tokens=False)
        input_ids      = full["input_ids"]
        attention_mask = [1] * len(input_ids)
        labels = [
            -100 if i < prompt_len else tok
            for i, tok in enumerate(input_ids)
        ]

        if len(input_ids) > MAX_LENGTH:
            input_ids      = input_ids[:MAX_LENGTH]
            attention_mask = attention_mask[:MAX_LENGTH]
            labels         = labels[:MAX_LENGTH]
        else:
            pad_len        = MAX_LENGTH - len(input_ids)
            input_ids      += [tokenizer.pad_token_id] * pad_len
            attention_mask += [0] * pad_len
            labels         += [-100] * pad_len

        true_label = "positive" if label == 1 else "negative"

        input_ids_list.append(input_ids)
        attention_masks_list.append(attention_mask)
        labels_list.append(labels)
        true_labels_list.append(true_label)
        texts_list.append(review)

    return Dataset.from_dict({
        "input_ids":      input_ids_list,
        "attention_mask": attention_masks_list,
        "labels":         labels_list,
        "true_label":     true_labels_list,
        "texts":          texts_list,
    })


In [8]:
# Model tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [9]:
def fits_model(example):

    text = build_full_text(
        example["text"],
        example["label"]
    )

    tokens = tokenizer(
        text,
        add_special_tokens=False
    )["input_ids"]

    return len(tokens) <= MAX_MODEL_LENGTH

In [ ]:
BATCH_SIZE  = 1
N_PER_CLASS = 100  # 100 neg + 100 pos = 200 total

filtered_test = dataset["test"].filter(fits_model)
test_dataset  = preprocess_dataset(filtered_test, tokenizer)

random.seed(42)
pos_idx  = [i for i in range(len(test_dataset)) if test_dataset[i]["true_label"] == "positive"]
neg_idx  = [i for i in range(len(test_dataset)) if test_dataset[i]["true_label"] == "negative"]
eval_idx = random.sample(pos_idx, N_PER_CLASS) + random.sample(neg_idx, N_PER_CLASS)

test_subset = test_dataset.select(eval_idx)
test_subset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False)

eval_texts  = [test_subset[i]["texts"]      for i in range(len(test_subset))]
eval_labels = [test_subset[i]["true_label"] for i in range(len(test_subset))]
print(f"Eval: {len(eval_idx)} examples | "
      f"{eval_labels.count('positive')} pos, {eval_labels.count('negative')} neg")


In [ ]:
CHECKPOINT_DIR = "../checkpoints/imdb"

config_11 = AutoConfig.from_pretrained(MODEL_NAME, local_files_only=True)
config_11.num_hidden_layers = 11


def load_model(n_layers, ckpt_name):
    if n_layers == 22:
        m = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, local_files_only=True, attn_implementation="eager"
        )
    else:
        m = AutoModelForCausalLM.from_config(config_11, attn_implementation="eager")
    m.load_state_dict(torch.load(f"{CHECKPOINT_DIR}/{ckpt_name}", map_location="cpu"))
    m.eval()
    return m


MODELS = {
    "teacher":       load_model(22, "best_teacher_model.pt"),
    "baseline":      load_model(11, "best_baseline_model.pt"),
    "bad_student":   load_model(11, "best_bad_student_model.pt"),
    "student":       load_model(11, "best_student_model.pt"),
    "student_local": load_model(11, "best_student_local_model.pt"),
}
MODEL_LABELS = {
    "teacher":       "P (teacher)",
    "baseline":      "B (no KD)",
    "bad_student":   "S_bad",
    "student":       "S1",
    "student_local": "S2",
}
print("Loaded:", list(MODELS.keys()))


# Métricas de comparación

In [ ]:
def frobenius_difference(
    x: torch.Tensor,
    y: torch.Tensor
):
    return torch.norm(
        x - y,
        p="fro"
    )


def cosine_similarity_tensor(
    x: torch.Tensor,
    y: torch.Tensor,
    eps: float = 1e-8
):
    x_flat = x.reshape(-1)
    y_flat = y.reshape(-1)

    similarity = F.cosine_similarity(
        x_flat.unsqueeze(0),
        y_flat.unsqueeze(0),
        dim=1,
        eps=eps
    )

    return similarity.squeeze()

def js_divergence_attention(
    attn_teacher: torch.Tensor,
    attn_student: torch.Tensor,
    eps: float = 1e-8
):
    # normalizar filas
    p = attn_teacher / (
        attn_teacher.sum(dim=-1, keepdim=True)
        + eps
    )

    q = attn_student / (
        attn_student.sum(dim=-1, keepdim=True)
        + eps
    )

    m = 0.5 * (p + q)

    kl_pm = torch.sum(
        p * torch.log(
            (p + eps) / (m + eps)
        ),
        dim=-1
    )

    kl_qm = torch.sum(
        q * torch.log(
            (q + eps) / (m + eps)
        ),
        dim=-1
    )

    jsd_rows = 0.5 * (
        kl_pm + kl_qm
    )

    return jsd_rows.mean()

def linear_cka(X, Y):
    """Linear CKA between activation matrices X:[n,d1] and Y:[n,d2]."""
    X = X.float();  Y = Y.float()
    n = X.shape[0]
    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)
    K = X @ X.T;  L = Y @ Y.T
    H = torch.eye(n, device=X.device) - torch.ones(n, n, device=X.device) / n
    Kc = H @ K @ H;  Lc = H @ L @ H
    hsic_kl = (Kc * Lc).sum()
    hsic_kk = (Kc * Kc).sum()
    hsic_ll = (Lc * Lc).sum()
    return (hsic_kl / (torch.sqrt(hsic_kk * hsic_ll) + 1e-10)).item()


def linear_cka(X, Y):
    """Linear CKA between activation matrices X:[n,d1] and Y:[n,d2]."""
    X = X.float();  Y = Y.float()
    n = X.shape[0]
    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)
    K = X @ X.T;  L = Y @ Y.T
    H = torch.eye(n, device=X.device) - torch.ones(n, n, device=X.device) / n
    Kc = H @ K @ H;  Lc = H @ L @ H
    hsic_kl = (Kc * Lc).sum()
    hsic_kk = (Kc * Kc).sum()
    hsic_ll = (Lc * Lc).sum()
    return (hsic_kl / (torch.sqrt(hsic_kk * hsic_ll) + 1e-10)).item()


# Rango efectivo

In [13]:
def effective_rank_participation_ratio(
    x: torch.Tensor,
    eps: float = 1e-12
):

    x = x.float()

    singular_values = (
        torch.linalg.svdvals(x)
    )

    power = singular_values**2

    numerator = (
        power.sum()**2
    )

    denominator = (
        (power**2).sum()
        + eps
    )

    rank_eff = (
        numerator
        / denominator
    )

    return rank_eff


def effective_rank_entropy(
    x: torch.Tensor,
    eps: float = 1e-12
):

    x = x.float()

    singular_values = (
        torch.linalg.svdvals(x)
    )

    p = singular_values / (
        singular_values.sum()
        + eps
    )

    entropy = -torch.sum(
        p * torch.log(p + eps)
    )

    rank_eff = torch.exp(entropy)

    return rank_eff

# Atención promedio

In [14]:
def compute_average_attentions(
    model,
    dataloader,
    device,
    max_batches=None,
    print_every=50
):

    model.eval()

    n_layers = (
        model.config.num_hidden_layers
    )

    attention_sums = [

        torch.zeros(
            (2048, 2048),
            dtype=torch.float32,
            device="cpu"
        )

        for _ in range(n_layers)
    ]

    n_examples = 0

    with torch.no_grad():

        for batch_idx, batch in enumerate(
            dataloader
        ):

            if (
                max_batches is not None
                and batch_idx >= max_batches
            ):
                break

            input_ids = batch[
                "input_ids"
            ].to(device)

            attention_mask = batch[
                "attention_mask"
            ].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_attentions=True
            )

            attentions = (
                outputs.attentions
            )

            batch_size = (
                input_ids.size(0)
            )

            for layer_idx in range(
                n_layers
            ):

                attn = attentions[
                    layer_idx
                ]

                # (B, H, S, S)
                # promedio heads
                attn = attn.mean(
                    dim=1
                )

                # (B, S, S)
                attn = (
                    attn
                    .float()
                    .cpu()
                )

                # suma batch
                attn_sum = attn.sum(
                    dim=0
                )

                attention_sums[
                    layer_idx
                ] += attn_sum

            n_examples += batch_size

            if (
                batch_idx
                % print_every
                == 0
            ):

                print(
                    f"Batch "
                    f"{batch_idx} | "
                    f"Examples "
                    f"{n_examples}"
                )

            del (
                input_ids,
                attention_mask,
                outputs,
                attentions,
                attn
            )

            gc.collect()

            torch.cuda.empty_cache()

    attention_means = [

        attn_sum / n_examples

        for attn_sum
        in attention_sums
    ]

    return attention_means

In [ ]:
def compute_hidden_states(model, prompts, tokenizer, device, n=100, max_seq=512):
    """
    Returns [n_layers+1, n, hidden_dim] – last real-token hidden vector per layer.
    """
    model.eval()
    all_hidden = []
    with torch.no_grad():
        for prompt in prompts[:n]:
            enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                            max_length=max_seq, add_special_tokens=False)
            input_ids = enc["input_ids"].to(device)
            attn_mask = enc["attention_mask"].to(device)
            last_idx  = int(attn_mask[0].nonzero()[-1].item())
            out = model(input_ids=input_ids, attention_mask=attn_mask,
                        output_hidden_states=True)
            hidden = torch.stack(
                [h[0, last_idx, :] for h in out.hidden_states]
            ).cpu()  # [n_layers+1, hidden_dim]
            all_hidden.append(hidden)
            del out;  torch.cuda.empty_cache()
    return torch.stack(all_hidden, dim=1)  # [n_layers+1, n, hidden_dim]


def compute_head_avg_patterns(model, prompts, tokenizer, device, n=50, max_seq=128):
    """
    Returns [n_layers, n_heads, max_seq] – avg attention from last token per head.
    """
    model.eval()
    n_layers = model.config.num_hidden_layers
    n_heads  = model.config.num_attention_heads
    sums  = torch.zeros(n_layers, n_heads, max_seq, dtype=torch.float32)
    count = 0
    with torch.no_grad():
        for prompt in prompts[:n]:
            enc = tokenizer(prompt, return_tensors="pt", truncation=True,
                            max_length=max_seq, add_special_tokens=False)
            input_ids = enc["input_ids"].to(device)
            attn_mask = enc["attention_mask"].to(device)
            seq_len   = input_ids.size(1)
            last_idx  = int(attn_mask[0].nonzero()[-1].item())
            out = model(input_ids=input_ids, attention_mask=attn_mask,
                        output_attentions=True)
            for l in range(n_layers):
                # [1, n_heads, S, S] -> [n_heads, seq_len]
                ha = out.attentions[l][0, :, last_idx, :seq_len].float().cpu()
                padded = torch.zeros(n_heads, max_seq)
                padded[:, :seq_len] = ha
                sums[l] += padded
            count += 1
            del out;  torch.cuda.empty_cache()
    return sums / count  # [n_layers, n_heads, max_seq]


## Teacher

In [ ]:
# Central attention computation for all 5 models
model_attentions = {}
for name, model in MODELS.items():
    print(f"Computing attentions for {MODEL_LABELS[name]}...")
    model.to(device)
    model_attentions[name] = compute_average_attentions(model, test_loader, device)
    model.cpu()
    torch.cuda.empty_cache()
    gc.collect()

# 22-layer teacher attentions (needed by teacher_avg_attentions + teacher_rollouts)
teacher_attentions = model_attentions["teacher"]

# Aliases — existing pairwise comparison cells use these names unchanged
baseline_attentions      = model_attentions["baseline"]
bad_student_attentions   = model_attentions["bad_student"]
student_attentions       = model_attentions["student"]
student_local_attentions = model_attentions["student_local"]

print("All attention maps computed.")


In [19]:
teacher_avg_attentions = [
    (teacher_attentions[2*i] + teacher_attentions[2*i + 1]) / 2
    for i in range(11)
]

In [20]:
def rollout_pair(A1, A2):
    I = torch.eye(A1.shape[-1], device=A1.device)
    
    # agregar residual y renormalizar
    A1 = A1 + I
    A1 = A1 / A1.sum(dim=-1, keepdim=True)
    
    A2 = A2 + I
    A2 = A2 / A2.sum(dim=-1, keepdim=True)
    
    return A1 @ A2

teacher_rollouts = [rollout_pair(teacher_attentions[2*i], teacher_attentions[2*i+1]) for i in range(11)]

# Teacher - Baseline comparison

In [25]:
# Cosine similarity entre attention maps
print("Similitud coseno con promedio de atenciones")
for i, (battn, tattn) in enumerate(zip(baseline_attentions, teacher_avg_attentions)):
    print(cosine_similarity_tensor(battn, tattn))

print("\n")

print("Similitud coseno con Attention Rollout")
for i, (battn, tattn) in enumerate(zip(baseline_attentions, teacher_rollouts)):
    print(cosine_similarity_tensor(battn, tattn))

Similitud coseno con promedio de atenciones
tensor(0.9421)
tensor(0.1868)
tensor(0.3199)
tensor(0.2535)
tensor(0.1633)
tensor(0.1885)
tensor(0.1674)
tensor(0.1357)
tensor(0.1439)
tensor(0.1589)
tensor(0.1368)


Similitud coseno con Attention Rollout
tensor(0.2627)
tensor(0.1744)
tensor(0.3039)
tensor(0.2479)
tensor(0.1505)
tensor(0.1637)
tensor(0.1300)
tensor(0.1086)
tensor(0.1164)
tensor(0.1309)
tensor(0.1142)


In [26]:
# Frobenius distance entre attention maps
print("Distancia Frobenius con promedio de atenciones")
for i, (battn, tattn) in enumerate(zip(baseline_attentions, teacher_avg_attentions)):
    print(frobenius_difference(battn, tattn))

print("\n")

print("Distancia Frobenius con Attention Rollout")
for i, (battn, tattn) in enumerate(zip(baseline_attentions, teacher_rollouts)):
    print(frobenius_difference(battn, tattn))

Distancia Frobenius con promedio de atenciones
tensor(1.2003)
tensor(19.3078)
tensor(36.1736)
tensor(39.2016)
tensor(25.9890)
tensor(22.3671)
tensor(18.0805)
tensor(17.8956)
tensor(19.4500)
tensor(20.1882)
tensor(21.1988)


Distancia Frobenius con Attention Rollout
tensor(11.4203)
tensor(21.8764)
tensor(30.7704)
tensor(32.3881)
tensor(25.2343)
tensor(23.0846)
tensor(20.3114)
tensor(20.4663)
tensor(21.3754)
tensor(21.7607)
tensor(22.4692)


In [27]:
# JS local
print("JS promediada por fila con promedio de atenciones")
for i, (battn, tattn) in enumerate(zip(baseline_attentions, teacher_avg_attentions)):
    print(js_divergence_attention(battn, tattn))

print("\n")

print("JS promediada por fila con Attention Rollout")
for i, (battn, tattn) in enumerate(zip(baseline_attentions, teacher_rollouts)):
    print(js_divergence_attention(battn, tattn))

JS promediada por fila con promedio de atenciones
tensor(0.0089)
tensor(0.2008)
tensor(0.4167)
tensor(0.4875)
tensor(0.3071)
tensor(0.2832)
tensor(0.2753)
tensor(0.2917)
tensor(0.3503)
tensor(0.3310)
tensor(0.3363)


JS promediada por fila con Attention Rollout
tensor(0.1049)
tensor(0.3211)
tensor(0.4924)
tensor(0.5432)
tensor(0.4017)
tensor(0.3751)
tensor(0.3574)
tensor(0.3727)
tensor(0.4208)
tensor(0.4055)
tensor(0.4120)


In [28]:
# Participation ratio
print("Participation ratio")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (battn, tattn, tattn2) in enumerate(zip(baseline_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_participation_ratio(battn):4f} | {effective_rank_participation_ratio(tattn):4f} | {effective_rank_participation_ratio(tattn2):4f}")

Participation ratio
index | student | teacher_avg | teacher_rollout
-----------------------------------------------
0 | 2.411058 | 2.976497 | 365.103943
1 | 4.373835 | 1.009933 | 1.848104
2 | 3.016362 | 1.000345 | 1.313450
3 | 2.694839 | 1.000532 | 1.285063
4 | 4.492360 | 1.003065 | 1.561989
5 | 5.041432 | 1.003987 | 1.720240
6 | 6.540986 | 1.010857 | 2.138787
7 | 11.732677 | 1.012277 | 2.154619
8 | 10.414304 | 1.008518 | 1.975988
9 | 8.078257 | 1.005468 | 1.900517
10 | 8.770850 | 1.012820 | 1.873872


In [29]:
# Entropy based effective rank
print("Entropy based effective rank")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (battn, tattn, tattn2) in enumerate(zip(baseline_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_entropy(battn):4f} | {effective_rank_entropy(tattn):4f} | {effective_rank_entropy(tattn2):4f}")

Entropy based effective rank
index | student | teacher_avg | teacher_rollout
-----------------------------------------------
0 | 47.817566 | 257.761902 | 2026.011475
1 | 256.266266 | 32.704948 | 1817.481567
2 | 223.644775 | 4.429290 | 1666.999878
3 | 273.882080 | 5.763463 | 1648.075317
4 | 255.872955 | 15.971895 | 1764.540405
5 | 297.750305 | 20.124546 | 1798.460205
6 | 339.967407 | 50.650951 | 1849.495850
7 | 428.148132 | 64.908333 | 1850.665649
8 | 435.651154 | 49.009342 | 1833.609009
9 | 394.209686 | 34.759365 | 1825.231079
10 | 422.027557 | 100.716484 | 1819.122925


# Teacher - Bad Student comparison

In [34]:
# Cosine similarity entre attention maps
print("Similitud coseno con promedio de atenciones")
for i, (bsattn, tattn) in enumerate(zip(bad_student_attentions, teacher_avg_attentions)):
    print(cosine_similarity_tensor(bsattn, tattn))

print("\n")

print("Similitud coseno con Attention Rollout")
for i, (bsattn, tattn) in enumerate(zip(bad_student_attentions, teacher_rollouts)):
    print(cosine_similarity_tensor(bsattn, tattn))

Similitud coseno con promedio de atenciones
tensor(0.9392)
tensor(0.6394)
tensor(0.4590)
tensor(0.2511)
tensor(0.3766)
tensor(0.1525)
tensor(0.2943)
tensor(0.2671)
tensor(0.1875)
tensor(0.2334)
tensor(0.2354)


Similitud coseno con Attention Rollout
tensor(0.2832)
tensor(0.5571)
tensor(0.4549)
tensor(0.2585)
tensor(0.3824)
tensor(0.1778)
tensor(0.3083)
tensor(0.2981)
tensor(0.2603)
tensor(0.3040)
tensor(0.2861)


In [35]:
# Frobenius distance entre attention maps
print("Distancia Frobenius con promedio de atenciones")
for i, (bsattn, tattn) in enumerate(zip(bad_student_attentions, teacher_avg_attentions)):
    print(frobenius_difference(bsattn, tattn))

print("\n")

print("Distancia Frobenius con Attention Rollout")
for i, (bsattn, tattn) in enumerate(zip(bad_student_attentions, teacher_rollouts)):
    print(frobenius_difference(bsattn, tattn))

Distancia Frobenius con promedio de atenciones
tensor(1.2332)
tensor(17.1487)
tensor(34.6945)
tensor(39.2212)
tensor(25.0266)
tensor(22.6618)
tensor(17.4976)
tensor(17.2242)
tensor(19.2340)
tensor(19.8381)
tensor(20.7307)


Distancia Frobenius con Attention Rollout
tensor(11.3482)
tensor(20.0769)
tensor(29.3116)
tensor(32.3503)
tensor(24.1930)
tensor(23.1073)
tensor(19.4879)
tensor(19.5070)
tensor(20.6585)
tensor(20.9136)
tensor(21.6597)


In [36]:
# JS local
print("JS promediada por fila con promedio de atenciones")
for i, (bsattn, tattn) in enumerate(zip(bad_student_attentions, teacher_avg_attentions)):
    print(js_divergence_attention(bsattn, tattn))

print("\n")

print("JS promediada por fila con Attention Rollout")
for i, (bsattn, tattn) in enumerate(zip(bad_student_attentions, teacher_rollouts)):
    print(js_divergence_attention(bsattn, tattn))

JS promediada por fila con promedio de atenciones
tensor(0.0100)
tensor(0.1193)
tensor(0.3573)
tensor(0.4728)
tensor(0.2343)
tensor(0.2419)
tensor(0.1780)
tensor(0.1804)
tensor(0.2078)
tensor(0.2123)
tensor(0.2239)


JS promediada por fila con Attention Rollout
tensor(0.1054)
tensor(0.2464)
tensor(0.4268)
tensor(0.5291)
tensor(0.3311)
tensor(0.3285)
tensor(0.2631)
tensor(0.2673)
tensor(0.2885)
tensor(0.2896)
tensor(0.3090)


In [37]:
# Participation ratio
print("Participation ratio")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (bsattn, tattn, tattn2) in enumerate(zip(bad_student_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_participation_ratio(bsattn):4f} | {effective_rank_participation_ratio(tattn):4f} | {effective_rank_participation_ratio(tattn2):4f}")

Participation ratio
index | student | teacher_avg | teacher_rollout
-----------------------------------------------
0 | 2.459714 | 2.976497 | 365.103943
1 | 1.763889 | 1.009933 | 1.848104
2 | 1.514616 | 1.000345 | 1.313450
3 | 2.328311 | 1.000532 | 1.285063
4 | 2.921611 | 1.003065 | 1.561989
5 | 5.290998 | 1.003987 | 1.720240
6 | 5.366787 | 1.010857 | 2.138787
7 | 4.684858 | 1.012277 | 2.154619
8 | 13.475297 | 1.008518 | 1.975988
9 | 10.792179 | 1.005468 | 1.900517
10 | 5.001162 | 1.012820 | 1.873872


In [38]:
# Entropy based effective rank
print("Entropy based effective rank")
print("index | student | teacher_avg | teacher_rollout")
print("-----------------------------------------------")
for i, (bsattn, tattn, tattn2) in enumerate(zip(bad_student_attentions, teacher_avg_attentions, teacher_rollouts)):
    print(f"{i} | {effective_rank_entropy(bsattn):4f} | {effective_rank_entropy(tattn):4f} | {effective_rank_entropy(tattn2):4f}")

Entropy based effective rank
index | student | teacher_avg | teacher_rollout
-----------------------------------------------
0 | 82.277687 | 257.761902 | 2026.011475
1 | 124.058304 | 32.704948 | 1817.481567
2 | 240.892136 | 4.429290 | 1666.999878
3 | 218.136780 | 5.763463 | 1648.075317
4 | 317.462524 | 15.971895 | 1764.540405
5 | 510.092682 | 20.124546 | 1798.460205
6 | 384.706543 | 50.650951 | 1849.495850
7 | 390.475891 | 64.908333 | 1850.665649
8 | 464.989807 | 49.009342 | 1833.609009
9 | 457.958069 | 34.759365 | 1825.231079
10 | 429.298462 | 100.716484 | 1819.122925


## Teacher - Student comparison

In [43]:
# Cosine similarity entre attention maps
print("Similitud coseno con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_avg_attentions)):
    print(cosine_similarity_tensor(sattn, tattn))

print("\n")

print("Similitud coseno con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_rollouts)):
    print(cosine_similarity_tensor(sattn, tattn))

Similitud coseno con promedio de atenciones
tensor(0.9271)
tensor(0.2688)
tensor(0.3348)
tensor(0.1360)
tensor(0.1623)
tensor(0.1579)
tensor(0.1731)
tensor(0.1687)
tensor(0.2197)
tensor(0.2192)
tensor(0.3066)


Similitud coseno con Attention Rollout
tensor(0.2752)
tensor(0.2268)
tensor(0.3322)
tensor(0.1484)
tensor(0.1930)
tensor(0.1914)
tensor(0.2138)
tensor(0.2225)
tensor(0.2439)
tensor(0.2334)
tensor(0.3291)


In [44]:
# Frobenius distance entre attention maps
print("Distancia Frobenius con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_avg_attentions)):
    print(frobenius_difference(sattn, tattn))

print("\n")

print("Distancia Frobenius con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_rollouts)):
    print(frobenius_difference(sattn, tattn))

Distancia Frobenius con promedio de atenciones
tensor(1.3432)
tensor(18.9485)
tensor(35.8680)
tensor(39.6938)
tensor(25.9925)
tensor(22.6046)
tensor(18.0666)
tensor(17.6782)
tensor(19.0674)
tensor(19.9193)
tensor(20.4072)


Distancia Frobenius con Attention Rollout
tensor(11.3760)
tensor(21.6454)
tensor(30.4318)
tensor(32.8054)
tensor(25.0566)
tensor(23.0084)
tensor(19.9416)
tensor(19.8836)
tensor(20.7777)
tensor(21.3191)
tensor(21.4697)


In [45]:
# JS local
print("JS promediada por fila con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_avg_attentions)):
    print(js_divergence_attention(sattn, tattn))

print("\n")

print("JS promediada por fila con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_attentions, teacher_rollouts)):
    print(js_divergence_attention(sattn, tattn))

JS promediada por fila con promedio de atenciones
tensor(0.0122)
tensor(0.1646)
tensor(0.3903)
tensor(0.4936)
tensor(0.2674)
tensor(0.2390)
tensor(0.1921)
tensor(0.1894)
tensor(0.2114)
tensor(0.2218)
tensor(0.2148)


JS promediada por fila con Attention Rollout
tensor(0.1108)
tensor(0.2943)
tensor(0.4644)
tensor(0.5489)
tensor(0.3641)
tensor(0.3247)
tensor(0.2791)
tensor(0.2791)
tensor(0.3001)
tensor(0.3124)
tensor(0.3020)


## Teacher - Student local comparison

In [50]:
# Cosine similarity entre attention maps
print("Similitud coseno con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_avg_attentions)):
    print(cosine_similarity_tensor(sattn, tattn))

print("\n")

print("Similitud coseno con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_rollouts)):
    print(cosine_similarity_tensor(sattn, tattn))

Similitud coseno con promedio de atenciones
tensor(0.9530)
tensor(0.3106)
tensor(0.5676)
tensor(0.3265)
tensor(0.2454)
tensor(0.3558)
tensor(0.2092)
tensor(0.1995)
tensor(0.2492)
tensor(0.1810)
tensor(0.2851)


Similitud coseno con Attention Rollout
tensor(0.2877)
tensor(0.2588)
tensor(0.5316)
tensor(0.3308)
tensor(0.2537)
tensor(0.3669)
tensor(0.2390)
tensor(0.2619)
tensor(0.2897)
tensor(0.2456)
tensor(0.3415)


In [51]:
# Frobenius distance entre attention maps
print("Distancia Frobenius con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_avg_attentions)):
    print(frobenius_difference(sattn, tattn))

print("\n")

print("Distancia Frobenius con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_rollouts)):
    print(frobenius_difference(sattn, tattn))

Distancia Frobenius con promedio de atenciones
tensor(1.0846)
tensor(18.7224)
tensor(33.7373)
tensor(38.8643)
tensor(25.6862)
tensor(21.5705)
tensor(17.9186)
tensor(17.5799)
tensor(18.9334)
tensor(20.0786)
tensor(20.4738)


Distancia Frobenius con Attention Rollout
tensor(11.3316)
tensor(21.4776)
tensor(28.6160)
tensor(32.0087)
tensor(24.8451)
tensor(22.1283)
tensor(19.8215)
tensor(19.6842)
tensor(20.5559)
tensor(21.2283)
tensor(21.3467)


In [52]:
# JS local
print("JS promediada por fila con promedio de atenciones")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_avg_attentions)):
    print(js_divergence_attention(sattn, tattn))

print("\n")

print("JS promediada por fila con Attention Rollout")
for i, (sattn, tattn) in enumerate(zip(student_local_attentions, teacher_rollouts)):
    print(js_divergence_attention(sattn, tattn))

JS promediada por fila con promedio de atenciones
tensor(0.0086)
tensor(0.1597)
tensor(0.3406)
tensor(0.4627)
tensor(0.2544)
tensor(0.2040)
tensor(0.1812)
tensor(0.1827)
tensor(0.2033)
tensor(0.2221)
tensor(0.2092)


JS promediada por fila con Attention Rollout
tensor(0.1042)
tensor(0.2893)
tensor(0.4227)
tensor(0.5173)
tensor(0.3564)
tensor(0.2960)
tensor(0.2689)
tensor(0.2678)
tensor(0.2884)
tensor(0.3035)
tensor(0.2924)


# Baseline - Student comparison

In [53]:
# Cosine similarity entre attention maps
print("Similitud coseno entre atenciones")
for i, (battn, sattn) in enumerate(zip(baseline_attentions, student_attentions)):
    print(cosine_similarity_tensor(battn, sattn))

Similitud coseno entre atenciones
tensor(0.9616)
tensor(0.6824)
tensor(0.5661)
tensor(0.6975)
tensor(0.6925)
tensor(0.4581)
tensor(0.5421)
tensor(0.4837)
tensor(0.5617)
tensor(0.6466)
tensor(0.5325)


In [54]:
# Frobenius distance entre attention maps
print("Distancia Frobenius entre atenciones")
for i, (battn, sattn) in enumerate(zip(baseline_attentions, student_attentions)):
    print(frobenius_difference(battn, sattn))

Distancia Frobenius entre atenciones
tensor(0.9490)
tensor(3.3461)
tensor(4.7210)
tensor(3.3066)
tensor(3.1945)
tensor(5.4268)
tensor(4.2504)
tensor(4.8470)
tensor(4.2260)
tensor(3.5969)
tensor(4.4178)


In [55]:
# JS local
print("JS promediada por fila entre atenciones")
for i, (battn, sattn) in enumerate(zip(baseline_attentions, student_attentions)):
    print(js_divergence_attention(battn, sattn))

JS promediada por fila entre atenciones
tensor(0.0076)
tensor(0.0723)
tensor(0.1065)
tensor(0.0873)
tensor(0.0811)
tensor(0.1126)
tensor(0.1242)
tensor(0.1410)
tensor(0.1936)
tensor(0.1579)
tensor(0.1724)


# Baseline - Student local comparison

In [56]:
# Cosine similarity entre attention maps
print("Similitud coseno entre atenciones")
for i, (battn, slattn) in enumerate(zip(baseline_attentions, student_local_attentions)):
    print(cosine_similarity_tensor(battn, slattn))

Similitud coseno entre atenciones
tensor(0.9914)
tensor(0.6275)
tensor(0.5592)
tensor(0.7397)
tensor(0.7749)
tensor(0.6011)
tensor(0.4805)
tensor(0.4078)
tensor(0.5124)
tensor(0.5131)
tensor(0.4587)


In [57]:
# Frobenius distance entre attention maps
print("Distancia Frobenius entre atenciones")
for i, (battn, slattn) in enumerate(zip(baseline_attentions, student_local_attentions)):
    print(frobenius_difference(battn, slattn))

Distancia Frobenius entre atenciones
tensor(0.4581)
tensor(3.8703)
tensor(6.0519)
tensor(3.0508)
tensor(2.6157)
tensor(3.9213)
tensor(4.7317)
tensor(5.4996)
tensor(4.5666)
tensor(4.4776)
tensor(4.9880)


In [58]:
# JS local
print("JS promediada por fila entre atenciones")
for i, (battn, slattn) in enumerate(zip(baseline_attentions, student_local_attentions)):
    print(js_divergence_attention(battn, slattn))


JS promediada por fila entre atenciones
tensor(0.0015)
tensor(0.0718)
tensor(0.1157)
tensor(0.0903)
tensor(0.0713)
tensor(0.1002)
tensor(0.1343)
tensor(0.1512)
tensor(0.1950)
tensor(0.1664)
tensor(0.1883)


# Baseline - Bad Student comparison

In [59]:
# Cosine similarity entre attention maps
print("Similitud coseno entre atenciones")
for i, (battn, bsattn) in enumerate(zip(baseline_attentions, bad_student_attentions)):
    print(cosine_similarity_tensor(battn, bsattn))

Similitud coseno entre atenciones
tensor(0.9622)
tensor(0.7259)
tensor(0.5293)
tensor(0.7421)
tensor(0.6745)
tensor(0.4456)
tensor(0.5350)
tensor(0.4452)
tensor(0.4294)
tensor(0.4541)
tensor(0.5156)


In [60]:
# Frobenius distance entre attention maps
print("Distancia Frobenius entre atenciones")
for i, (battn, bsattn) in enumerate(zip(baseline_attentions, bad_student_attentions)):
    print(frobenius_difference(battn, bsattn))

Distancia Frobenius entre atenciones
tensor(0.9514)
tensor(3.1302)
tensor(5.9543)
tensor(2.9733)
tensor(3.4063)
tensor(5.6465)
tensor(4.3556)
tensor(5.1892)
tensor(5.2279)
tensor(4.9968)
tensor(4.5380)


In [61]:
# JS local
print("JS promediada por fila entre atenciones")
for i, (battn, bsattn) in enumerate(zip(baseline_attentions, bad_student_attentions)):
    print(js_divergence_attention(battn, bsattn))

JS promediada por fila entre atenciones
tensor(0.0046)
tensor(0.0628)
tensor(0.1217)
tensor(0.0803)
tensor(0.0799)
tensor(0.1182)
tensor(0.1283)
tensor(0.1588)
tensor(0.2095)
tensor(0.1756)
tensor(0.1763)


# Bad Student - Student comparison

In [62]:
# Cosine similarity entre attention maps
print("Similitud coseno entre atenciones")
for i, (bsattn, sattn) in enumerate(zip(bad_student_attentions, student_attentions)):
    print(cosine_similarity_tensor(bsattn, sattn))

Similitud coseno entre atenciones
tensor(0.9709)
tensor(0.7337)
tensor(0.7824)
tensor(0.9115)
tensor(0.9157)
tensor(0.9239)
tensor(0.9327)
tensor(0.8805)
tensor(0.9237)
tensor(0.8414)
tensor(0.8975)


In [63]:
# Frobenius distance entre attention maps
print("Distancia Frobenius entre atenciones")
for i, (bsattn, sattn) in enumerate(zip(bad_student_attentions, student_attentions)):
    print(frobenius_difference(bsattn, sattn))

Distancia Frobenius entre atenciones
tensor(0.8372)
tensor(3.2452)
tensor(4.3429)
tensor(1.7878)
tensor(1.7611)
tensor(2.3568)
tensor(1.6948)
tensor(2.3061)
tensor(1.9619)
tensor(2.7050)
tensor(2.0256)


In [64]:
# JS local
print("JS promediada por fila entre atenciones")
for i, (bsattn, sattn) in enumerate(zip(bad_student_attentions, student_attentions)):
    print(js_divergence_attention(bsattn, sattn))

JS promediada por fila entre atenciones
tensor(0.0053)
tensor(0.0279)
tensor(0.0445)
tensor(0.0147)
tensor(0.0176)
tensor(0.0231)
tensor(0.0188)
tensor(0.0248)
tensor(0.0191)
tensor(0.0277)
tensor(0.0303)


# Bad Student - Student local comparison

In [65]:
# Cosine similarity entre attention maps
print("Similitud coseno entre atenciones")
for i, (bsattn, slattn) in enumerate(zip(bad_student_attentions, student_local_attentions)):
    print(cosine_similarity_tensor(bsattn, slattn))

Similitud coseno entre atenciones
tensor(0.9762)
tensor(0.7301)
tensor(0.7403)
tensor(0.8952)
tensor(0.9076)
tensor(0.8543)
tensor(0.9200)
tensor(0.8769)
tensor(0.9522)
tensor(0.9566)
tensor(0.8996)


In [66]:
# Frobenius distance entre attention maps
print("Distancia Frobenius entre atenciones")
for i, (bsattn, slattn) in enumerate(zip(bad_student_attentions, student_local_attentions)):
    print(frobenius_difference(bsattn, slattn))

Distancia Frobenius entre atenciones
tensor(0.7585)
tensor(3.4450)
tensor(5.1487)
tensor(1.9284)
tensor(1.8572)
tensor(3.2375)
tensor(1.9268)
tensor(2.4787)
tensor(1.5447)
tensor(1.4695)
tensor(2.1382)


In [67]:
# JS local
print("JS promediada por fila entre atenciones")
for i, (bsattn, slattn) in enumerate(zip(bad_student_attentions, student_local_attentions)):
    print(js_divergence_attention(bsattn, slattn))

JS promediada por fila entre atenciones
tensor(0.0034)
tensor(0.0265)
tensor(0.0610)
tensor(0.0287)
tensor(0.0189)
tensor(0.0329)
tensor(0.0222)
tensor(0.0251)
tensor(0.0203)
tensor(0.0216)
tensor(0.0330)


# Student - Student comparison

In [68]:
# Cosine similarity entre attention maps
print("Similitud coseno entre atenciones")
for i, (sattn, slattn) in enumerate(zip(student_attentions, student_local_attentions)):
    print(cosine_similarity_tensor(sattn, slattn))

Similitud coseno entre atenciones
tensor(0.9713)
tensor(0.9258)
tensor(0.8185)
tensor(0.8940)
tensor(0.9451)
tensor(0.9049)
tensor(0.9586)
tensor(0.9706)
tensor(0.9590)
tensor(0.8860)
tensor(0.8624)


In [69]:
# Frobenius distance entre attention maps
print("Distancia Frobenius entre atenciones")
for i, (sattn, slattn) in enumerate(zip(student_attentions, student_local_attentions)):
    print(frobenius_difference(sattn, slattn))

Distancia Frobenius entre atenciones
tensor(0.8196)
tensor(1.8418)
tensor(4.2136)
tensor(1.9827)
tensor(1.3581)
tensor(2.6141)
tensor(1.4082)
tensor(1.3172)
tensor(1.2321)
tensor(2.0819)
tensor(2.4813)


In [70]:
# JS local
print("JS promediada por fila entre atenciones")
for i, (sattn, slattn) in enumerate(zip(student_attentions, student_local_attentions)):
    print(js_divergence_attention(sattn, slattn))

JS promediada por fila entre atenciones
tensor(0.0055)
tensor(0.0100)
tensor(0.0327)
tensor(0.0304)
tensor(0.0128)
tensor(0.0225)
tensor(0.0150)
tensor(0.0128)
tensor(0.0126)
tensor(0.0185)
tensor(0.0384)


# Effective ranks comparisons

In [71]:
# Participation ratio
print("Participation ratio")
print("index | teacher_avg | teacher_roll | baseline | bad_student |student | student_local")
print("-----------------------------------------------")
for i, (tattn, t2attn, battn, bsattn, sattn, slattn) in enumerate(zip(teacher_avg_attentions, teacher_rollouts, baseline_attentions, bad_student_attentions, student_attentions, student_local_attentions)):
    print(f"{i} | {effective_rank_participation_ratio(tattn):4f} | {effective_rank_participation_ratio(t2attn):4f} | {effective_rank_participation_ratio(battn):4f} | {effective_rank_participation_ratio(bsattn):4f} | {effective_rank_participation_ratio(sattn):4f} | {effective_rank_participation_ratio(slattn):4f} ")


Participation ratio
index | teacher_avg | teacher_roll | baseline | bad_student |student | student_local
-----------------------------------------------
0 | 2.976497 | 365.103943 | 2.411058 | 2.459714 | 2.435164 | 2.502691 
1 | 1.009933 | 1.848104 | 4.373835 | 1.763889 | 2.089699 | 1.789329 
2 | 1.000345 | 1.313450 | 3.016362 | 1.514616 | 1.875404 | 1.282771 
3 | 1.000532 | 1.285063 | 2.694839 | 2.328311 | 2.092130 | 2.153772 
4 | 1.003065 | 1.561989 | 4.492360 | 2.921611 | 4.126306 | 3.278673 
5 | 1.003987 | 1.720240 | 5.041432 | 5.290998 | 8.576671 | 4.792106 
6 | 1.010857 | 2.138787 | 6.540986 | 5.366787 | 9.733599 | 12.610973 
7 | 1.012277 | 2.154619 | 11.732677 | 4.684858 | 10.104803 | 15.858840 
8 | 1.008518 | 1.975988 | 10.414304 | 13.475297 | 4.953476 | 7.064105 
9 | 1.005468 | 1.900517 | 8.078257 | 10.792179 | 3.422758 | 8.693152 
10 | 1.012820 | 1.873872 | 8.770850 | 5.001162 | 3.270441 | 8.683012 


In [72]:
# Entropy based effective rank
print("Entropy based effective rank")
print("index | teacher_avg | teacher_roll | baseline | bad_student |student | student_local")
print("-----------------------------------------------")
for i, (tattn, t2attn, battn, bsattn, sattn, slattn) in enumerate(zip(teacher_avg_attentions, teacher_rollouts, baseline_attentions, bad_student_attentions, student_attentions, student_local_attentions)):
    print(f"{i} | {effective_rank_entropy(tattn):4f} | {effective_rank_entropy(t2attn):4f} | {effective_rank_entropy(battn):4f} | {effective_rank_entropy(bsattn):4f} | {effective_rank_entropy(sattn):4f} | {effective_rank_entropy(slattn):4f} ")

Entropy based effective rank
index | teacher_avg | teacher_roll | baseline | bad_student |student | student_local
-----------------------------------------------
0 | 257.761902 | 2026.011475 | 47.817566 | 82.277687 | 77.745987 | 93.271255 
1 | 32.704948 | 1817.481567 | 256.266266 | 124.058304 | 75.566002 | 56.276192 
2 | 4.429290 | 1666.999878 | 223.644775 | 240.892136 | 137.775803 | 17.545155 
3 | 5.763463 | 1648.075317 | 273.882080 | 218.136780 | 159.863037 | 207.461075 
4 | 15.971895 | 1764.540405 | 255.872955 | 317.462524 | 354.986694 | 259.767944 
5 | 20.124546 | 1798.460205 | 297.750305 | 510.092682 | 534.862976 | 396.525909 
6 | 50.650951 | 1849.495850 | 339.967407 | 384.706543 | 400.907806 | 495.884583 
7 | 64.908333 | 1850.665649 | 428.148132 | 390.475891 | 419.543579 | 457.309113 
8 | 49.009342 | 1833.609009 | 435.651154 | 464.989807 | 348.232544 | 414.098938 
9 | 34.759365 | 1825.231079 | 394.209686 | 457.958069 | 298.162170 | 428.959808 
10 | 100.716484 | 1819.122925 | 422.

# Comparación con promedio y rollout total

In [73]:
teacher_total_avg_attention = sum(teacher_attentions)/len(teacher_attentions)

def total_rollout(attentions):
    I = torch.eye(2048, device="cpu")
    R = None
    for A in attentions:

        attn = A + I
        attn = attn / attn.sum(dim=-1, keepdim=True)

        if R is None:
            R = attn
        else:
            R = R @ attn
    
    return R

teacher_total_rollout = total_rollout(teacher_attentions)

In [74]:
baseline_total_avg_attention = sum(baseline_attentions)/len(baseline_attentions)
baseline_total_rollout = total_rollout(baseline_attentions)

In [75]:
student_total_avg_attention = sum(student_attentions)/len(student_attentions)
student_total_rollout = total_rollout(student_attentions)

In [76]:
student_local_total_avg_attention = sum(student_local_attentions)/len(student_local_attentions)
student_local_total_rollout = total_rollout(student_local_attentions)

In [77]:
bad_student_total_avg_attention = sum(bad_student_attentions)/len(bad_student_attentions)
bad_student_total_rollout = total_rollout(bad_student_attentions)

In [78]:
# Cosine similarity entre attention maps
print("Similitud coseno")

print("Promedio student - Promedio teacher")
print(frobenius_difference(student_total_avg_attention, teacher_total_avg_attention))

print("Promedio student - Rollout teacher")
print(cosine_similarity_tensor(student_total_avg_attention, teacher_total_rollout))

print("Promedio student local - Promedio teacher")
print(cosine_similarity_tensor(student_local_total_avg_attention, teacher_total_avg_attention))

print("Promedio student local - Rollout teacher")
print(cosine_similarity_tensor(student_local_total_avg_attention, teacher_total_rollout))

print("Rollout student - Promedio teacher")
print(cosine_similarity_tensor(student_total_rollout, teacher_total_avg_attention))

print("Rollout student - Rollout teacher")
print(cosine_similarity_tensor(student_total_rollout, teacher_total_rollout))

print("Rollout student local - Promedio teacher")
print(cosine_similarity_tensor(student_local_total_rollout, teacher_total_avg_attention))

print("Rollout student local - Rollout teacher")
print(cosine_similarity_tensor(student_local_total_rollout, teacher_total_rollout))

print("Promedio student - Promedio student local")
print(cosine_similarity_tensor(student_total_avg_attention, student_local_total_avg_attention))

print("Promedio student - Rollout student local")
print(cosine_similarity_tensor(student_total_avg_attention, student_local_total_rollout))

print("Rollout student - Promedio student local")
print(cosine_similarity_tensor(student_total_rollout, student_local_total_avg_attention))

print("Rollout student - Rollout student local")
print(cosine_similarity_tensor(student_total_rollout, student_local_total_rollout))

print("Baseline promedio - Promedio teacher")
print(cosine_similarity_tensor(baseline_total_avg_attention, teacher_total_avg_attention))

print("Baseline promedio - Rollout teacher")
print(cosine_similarity_tensor(baseline_total_avg_attention, teacher_total_rollout))

print("Baseline promedio - Promedio student")
print(cosine_similarity_tensor(baseline_total_avg_attention, student_total_avg_attention))

print("Baseline promedio - Rollout student")
print(cosine_similarity_tensor(baseline_total_avg_attention, student_total_rollout))

print("Baseline promedio - Promedio student local")
print(cosine_similarity_tensor(baseline_total_avg_attention, student_local_total_avg_attention))

print("Baseline promedio - Rollout student local")
print(cosine_similarity_tensor(baseline_total_avg_attention, student_local_total_rollout))

print("Baseline promedio - Promedio bad student")
print(cosine_similarity_tensor(baseline_total_avg_attention, bad_student_total_avg_attention))

print("Baseline promedio - Rollout bad student")
print(cosine_similarity_tensor(baseline_total_avg_attention, bad_student_total_rollout))

print("Baseline rollout - Promedio teacher")
print(cosine_similarity_tensor(baseline_total_rollout, teacher_total_avg_attention))

print("Baseline rollout - Rollout teacher")
print(cosine_similarity_tensor(baseline_total_rollout, teacher_total_rollout))

print("Baseline rollout - Promedio student")
print(cosine_similarity_tensor(baseline_total_rollout, student_total_avg_attention))

print("Baseline rollout - Rollout student")
print(cosine_similarity_tensor(baseline_total_rollout, student_total_rollout))

print("Baseline rollout - Promedio student local")
print(cosine_similarity_tensor(baseline_total_rollout, student_local_total_avg_attention))

print("Baseline rollout - Rollout student local")
print(cosine_similarity_tensor(baseline_total_rollout, student_local_total_rollout))

print("Baseline rollout - Promedio bad student")
print(cosine_similarity_tensor(baseline_total_rollout, bad_student_total_avg_attention))

print("Baseline rollout - Rollout bad student")
print(cosine_similarity_tensor(baseline_total_rollout, bad_student_total_rollout))

print("Promedio bad student - Promedio teacher")
print(cosine_similarity_tensor(bad_student_total_avg_attention, teacher_total_avg_attention))

print("Promedio bad student - Rollout teacher")
print(cosine_similarity_tensor(bad_student_total_avg_attention, teacher_total_rollout))

print("Promedio bad student - Promedio student")
print(cosine_similarity_tensor(bad_student_total_avg_attention, student_total_avg_attention))

print("Promedio bad student - Rollout student")
print(cosine_similarity_tensor(bad_student_total_avg_attention, student_total_rollout))

print("Promedio bad student - Promedio student local")
print(cosine_similarity_tensor(bad_student_total_avg_attention, student_local_total_avg_attention))

print("Promedio bad student - Rollout student local")
print(cosine_similarity_tensor(bad_student_total_avg_attention, student_local_total_rollout))

print("Rollout bad student - Promedio teacher")
print(cosine_similarity_tensor(bad_student_total_rollout, teacher_total_avg_attention))

print("Rollout bad student - Rollout teacher")
print(cosine_similarity_tensor(bad_student_total_rollout, teacher_total_rollout))

print("Rollout bad student - Promedio student")
print(cosine_similarity_tensor(bad_student_total_rollout, student_total_avg_attention))

print("Rollout bad student - Rollout student")
print(cosine_similarity_tensor(bad_student_total_rollout, student_total_rollout))

print("Rollout bad student - Promedio student local")
print(cosine_similarity_tensor(bad_student_total_rollout, student_local_total_avg_attention))

print("Rollout bad student - Rollout student local")
print(cosine_similarity_tensor(bad_student_total_rollout, student_local_total_rollout))

Similitud coseno
Promedio student - Promedio teacher
tensor(21.4037)
Promedio student - Rollout teacher
tensor(0.1731)
Promedio student local - Promedio teacher
tensor(0.3478)
Promedio student local - Rollout teacher
tensor(0.2693)
Rollout student - Promedio teacher
tensor(0.9534)
Rollout student - Rollout teacher
tensor(0.9434)
Rollout student local - Promedio teacher
tensor(0.9773)
Rollout student local - Rollout teacher
tensor(0.9699)
Promedio student - Promedio student local
tensor(0.9865)
Promedio student - Rollout student local
tensor(0.2977)
Rollout student - Promedio student local
tensor(0.4221)
Rollout student - Rollout student local
tensor(0.9953)
Baseline promedio - Promedio teacher
tensor(0.2189)
Baseline promedio - Rollout teacher
tensor(0.1493)
Baseline promedio - Promedio student
tensor(0.7844)
Baseline promedio - Rollout student
tensor(0.2730)
Baseline promedio - Promedio student local
tensor(0.7703)
Baseline promedio - Rollout student local
tensor(0.2473)
Baseline prom

In [79]:
# Distancia Frobenius entre attention maps
print("Distancia Frobenius")

print("Promedio student - Promedio teacher")
print(frobenius_difference(student_total_avg_attention, teacher_total_avg_attention))

print("Promedio student - Rollout teacher")
print(frobenius_difference(student_total_avg_attention, teacher_total_rollout))

print("Promedio student local - Promedio teacher")
print(frobenius_difference(student_local_total_avg_attention, teacher_total_avg_attention))

print("Promedio student local - Rollout teacher")
print(frobenius_difference(student_local_total_avg_attention, teacher_total_rollout))

print("Rollout student - Promedio teacher")
print(frobenius_difference(student_total_rollout, teacher_total_avg_attention))

print("Rollout student - Rollout teacher")
print(frobenius_difference(student_total_rollout, teacher_total_rollout))

print("Rollout student local - Promedio teacher")
print(frobenius_difference(student_local_total_rollout, teacher_total_avg_attention))

print("Rollout student local - Rollout teacher")
print(frobenius_difference(student_local_total_rollout, teacher_total_rollout))

print("Promedio student - Promedio student local")
print(frobenius_difference(student_total_avg_attention, student_local_total_avg_attention))

print("Promedio student - Rollout student local")
print(frobenius_difference(student_total_avg_attention, student_local_total_rollout))

print("Rollout student - Promedio student local")
print(frobenius_difference(student_total_rollout, student_local_total_avg_attention))

print("Rollout student - Rollout student local")
print(frobenius_difference(student_total_rollout, student_local_total_rollout))

print("Baseline promedio - Promedio teacher")
print(frobenius_difference(baseline_total_avg_attention, teacher_total_avg_attention))

print("Baseline promedio - Rollout teacher")
print(frobenius_difference(baseline_total_avg_attention, teacher_total_rollout))

print("Baseline promedio - Promedio student")
print(frobenius_difference(baseline_total_avg_attention, student_total_avg_attention))

print("Baseline promedio - Rollout student")
print(frobenius_difference(baseline_total_avg_attention, student_total_rollout))

print("Baseline promedio - Promedio student local")
print(frobenius_difference(baseline_total_avg_attention, student_local_total_avg_attention))

print("Baseline promedio - Rollout student local")
print(frobenius_difference(baseline_total_avg_attention, student_local_total_rollout))

print("Baseline promedio - Promedio bad student")
print(frobenius_difference(baseline_total_avg_attention, bad_student_total_avg_attention))

print("Baseline promedio - Rollout bad student")
print(frobenius_difference(baseline_total_avg_attention, bad_student_total_rollout))

print("Baseline rollout - Promedio teacher")
print(frobenius_difference(baseline_total_rollout, teacher_total_avg_attention))

print("Baseline rollout - Rollout teacher")
print(frobenius_difference(baseline_total_rollout, teacher_total_rollout))

print("Baseline rollout - Promedio student")
print(frobenius_difference(baseline_total_rollout, student_total_avg_attention))

print("Baseline rollout - Rollout student")
print(frobenius_difference(baseline_total_rollout, student_total_rollout))

print("Baseline rollout - Promedio student local")
print(frobenius_difference(baseline_total_rollout, student_local_total_avg_attention))

print("Baseline rollout - Rollout student local")
print(frobenius_difference(baseline_total_rollout, student_local_total_rollout))

print("Baseline rollout - Promedio bad student")
print(frobenius_difference(baseline_total_rollout, bad_student_total_avg_attention))

print("Baseline rollout - Rollout bad student")
print(frobenius_difference(baseline_total_rollout, bad_student_total_rollout))

print("Promedio bad student - Promedio teacher")
print(frobenius_difference(bad_student_total_avg_attention, teacher_total_avg_attention))

print("Promedio bad student - Rollout teacher")
print(frobenius_difference(bad_student_total_avg_attention, teacher_total_rollout))

print("Promedio bad student - Promedio student")
print(frobenius_difference(bad_student_total_avg_attention, student_total_avg_attention))

print("Promedio bad student - Rollout student")
print(frobenius_difference(bad_student_total_avg_attention, student_total_rollout))

print("Promedio bad student - Promedio student local")
print(frobenius_difference(bad_student_total_avg_attention, student_local_total_avg_attention))

print("Promedio bad student - Rollout student local")
print(frobenius_difference(bad_student_total_avg_attention, student_local_total_rollout))

print("Rollout bad student - Promedio teacher")
print(frobenius_difference(bad_student_total_rollout, teacher_total_avg_attention))

print("Rollout bad student - Rollout teacher")
print(frobenius_difference(bad_student_total_rollout, teacher_total_rollout))

print("Rollout bad student - Promedio student")
print(frobenius_difference(bad_student_total_rollout, student_total_avg_attention))

print("Rollout bad student - Rollout student")
print(frobenius_difference(bad_student_total_rollout, student_total_rollout))

print("Rollout bad student - Promedio student local")
print(frobenius_difference(bad_student_total_rollout, student_local_total_avg_attention))

print("Rollout bad student - Rollout student local")
print(frobenius_difference(bad_student_total_rollout, student_local_total_rollout))

Distancia Frobenius
Promedio student - Promedio teacher
tensor(21.4037)
Promedio student - Rollout teacher
tensor(44.7235)
Promedio student local - Promedio teacher
tensor(20.9771)
Promedio student local - Rollout teacher
tensor(44.3146)
Rollout student - Promedio teacher
tensor(6.8544)
Rollout student - Rollout teacher
tensor(27.6911)
Rollout student local - Promedio teacher
tensor(4.7195)
Rollout student local - Rollout teacher
tensor(24.4101)
Promedio student - Promedio student local
tensor(0.6835)
Promedio student - Rollout student local
tensor(21.2566)
Rollout student - Promedio student local
tensor(18.0665)
Rollout student - Rollout student local
tensor(3.3768)
Baseline promedio - Promedio teacher
tensor(21.5417)
Baseline promedio - Rollout teacher
tensor(44.8384)
Baseline promedio - Promedio student
tensor(2.5221)
Baseline promedio - Rollout student
tensor(18.7407)
Baseline promedio - Promedio student local
tensor(2.6560)
Baseline promedio - Rollout student local
tensor(21.4874)

In [80]:
# JS local entre attention maps
print("JS local")

print("Promedio student - Promedio teacher")
print(js_divergence_attention(student_total_avg_attention, teacher_total_avg_attention))

print("Promedio student - Rollout teacher")
print(js_divergence_attention(student_total_avg_attention, teacher_total_rollout))

print("Promedio student local - Promedio teacher")
print(js_divergence_attention(student_local_total_avg_attention, teacher_total_avg_attention))

print("Promedio student local - Rollout teacher")
print(js_divergence_attention(student_local_total_avg_attention, teacher_total_rollout))

print("Rollout student - Promedio teacher")
print(js_divergence_attention(student_total_rollout, teacher_total_avg_attention))

print("Rollout student - Rollout teacher")
print(js_divergence_attention(student_total_rollout, teacher_total_rollout))

print("Rollout student local - Promedio teacher")
print(js_divergence_attention(student_local_total_rollout, teacher_total_avg_attention))

print("Rollout student local - Rollout teacher")
print(js_divergence_attention(student_local_total_rollout, teacher_total_rollout))

print("Promedio student - Promedio student local")
print(js_divergence_attention(student_total_avg_attention, student_local_total_avg_attention))

print("Promedio student - Rollout student local")
print(js_divergence_attention(student_total_avg_attention, student_local_total_rollout))

print("Rollout student - Promedio student local")
print(js_divergence_attention(student_total_rollout, student_local_total_avg_attention))

print("Rollout student - Rollout student local")
print(js_divergence_attention(student_total_rollout, student_local_total_rollout))

print("Baseline promedio - Promedio teacher")
print(js_divergence_attention(baseline_total_avg_attention, teacher_total_avg_attention))

print("Baseline promedio - Rollout teacher")
print(js_divergence_attention(baseline_total_avg_attention, teacher_total_rollout))

print("Baseline promedio - Promedio student")
print(js_divergence_attention(baseline_total_avg_attention, student_total_avg_attention))

print("Baseline promedio - Rollout student")
print(js_divergence_attention(baseline_total_avg_attention, student_total_rollout))

print("Baseline promedio - Promedio student local")
print(js_divergence_attention(baseline_total_avg_attention, student_local_total_avg_attention))

print("Baseline promedio - Rollout student local")
print(js_divergence_attention(baseline_total_avg_attention, student_local_total_rollout))

print("Baseline promedio - Promedio bad student")
print(js_divergence_attention(baseline_total_avg_attention, bad_student_total_avg_attention))

print("Baseline promedio - Rollout bad student")
print(js_divergence_attention(baseline_total_avg_attention, bad_student_total_rollout))

print("Baseline rollout - Promedio teacher")
print(js_divergence_attention(baseline_total_rollout, teacher_total_avg_attention))

print("Baseline rollout - Rollout teacher")
print(js_divergence_attention(baseline_total_rollout, teacher_total_rollout))

print("Baseline rollout - Promedio student")
print(js_divergence_attention(baseline_total_rollout, student_total_avg_attention))

print("Baseline rollout - Rollout student")
print(js_divergence_attention(baseline_total_rollout, student_total_rollout))

print("Baseline rollout - Promedio student local")
print(js_divergence_attention(baseline_total_rollout, student_local_total_avg_attention))

print("Baseline rollout - Rollout student local")
print(js_divergence_attention(baseline_total_rollout, student_local_total_rollout))

print("Baseline rollout - Promedio bad student")
print(js_divergence_attention(baseline_total_rollout, bad_student_total_avg_attention))

print("Baseline rollout - Rollout bad student")
print(js_divergence_attention(baseline_total_rollout, bad_student_total_rollout))

print("Promedio bad student - Promedio teacher")
print(js_divergence_attention(bad_student_total_avg_attention, teacher_total_avg_attention))

print("Promedio bad student - Rollout teacher")
print(js_divergence_attention(bad_student_total_avg_attention, teacher_total_rollout))

print("Promedio bad student - Promedio student")
print(js_divergence_attention(bad_student_total_avg_attention, student_total_avg_attention))

print("Promedio bad student - Rollout student")
print(js_divergence_attention(bad_student_total_avg_attention, student_total_rollout))

print("Promedio bad student - Promedio student local")
print(js_divergence_attention(bad_student_total_avg_attention, student_local_total_avg_attention))

print("Promedio bad student - Rollout student local")
print(js_divergence_attention(bad_student_total_avg_attention, student_local_total_rollout))

print("Rollout bad student - Promedio teacher")
print(js_divergence_attention(bad_student_total_rollout, teacher_total_avg_attention))

print("Rollout bad student - Rollout teacher")
print(js_divergence_attention(bad_student_total_rollout, teacher_total_rollout))

print("Rollout bad student - Promedio student")
print(js_divergence_attention(bad_student_total_rollout, student_total_avg_attention))

print("Rollout bad student - Rollout student")
print(js_divergence_attention(bad_student_total_rollout, student_total_rollout))

print("Rollout bad student - Promedio student local")
print(js_divergence_attention(bad_student_total_rollout, student_local_total_avg_attention))

print("Rollout bad student - Rollout student local")
print(js_divergence_attention(bad_student_total_rollout, student_local_total_rollout))

JS local
Promedio student - Promedio teacher
tensor(0.1946)
Promedio student - Rollout teacher
tensor(0.6554)
Promedio student local - Promedio teacher
tensor(0.1815)
Promedio student local - Rollout teacher
tensor(0.6374)
Rollout student - Promedio teacher
tensor(0.1787)
Rollout student - Rollout teacher
tensor(0.2719)
Rollout student local - Promedio teacher
tensor(0.1568)
Rollout student local - Rollout teacher
tensor(0.2306)
Promedio student - Promedio student local
tensor(0.0024)
Promedio student - Rollout student local
tensor(0.3120)
Rollout student - Promedio student local
tensor(0.2879)
Rollout student - Rollout student local
tensor(0.0030)
Baseline promedio - Promedio teacher
tensor(0.2254)
Baseline promedio - Rollout teacher
tensor(0.6629)
Baseline promedio - Promedio student
tensor(0.0392)
Baseline promedio - Rollout student
tensor(0.3075)
Baseline promedio - Promedio student local
tensor(0.0404)
Baseline promedio - Rollout student local
tensor(0.3193)
Baseline promedio - Pr

In [81]:
# Participation ratio
print("Participation ratio")
print("Teacher avg")
print(effective_rank_participation_ratio(teacher_total_avg_attention))
print("Student avg")
print(effective_rank_participation_ratio(student_total_avg_attention))
print("Student local avg")
print(effective_rank_participation_ratio(student_local_total_avg_attention))
print("Teacher rollout")
print(effective_rank_participation_ratio(teacher_total_rollout))
print("Student rollout")
print(effective_rank_participation_ratio(student_total_rollout))
print("Student local rollout")
print(effective_rank_participation_ratio(student_local_total_rollout))
print("Baseline avg")
print(effective_rank_participation_ratio(baseline_total_avg_attention))
print("Baseline avg")
print(effective_rank_participation_ratio(baseline_total_rollout))
print("Bad student avg")
print(effective_rank_participation_ratio(bad_student_total_avg_attention))
print("Bad student rollout")
print(effective_rank_participation_ratio(bad_student_total_rollout))

Participation ratio
Teacher avg
tensor(1.0052)
Student avg
tensor(3.2826)
Student local avg
tensor(3.2873)
Teacher rollout
tensor(1.)
Student rollout
tensor(1.0055)
Student local rollout
tensor(1.0022)
Baseline avg
tensor(3.1357)
Baseline avg
tensor(1.0019)
Bad student avg
tensor(3.3120)
Bad student rollout
tensor(1.0037)


In [82]:
# Entropy based effective rank
print("Entropy based effective rank")
print("Teacher avg")
print(effective_rank_entropy(teacher_total_avg_attention))
print("Student avg")
print(effective_rank_entropy(student_total_avg_attention))
print("Student local avg")
print(effective_rank_entropy(student_local_total_avg_attention))
print("Teacher rollout")
print(effective_rank_entropy(teacher_total_rollout))
print("Student rollout")
print(effective_rank_entropy(student_total_rollout))
print("Student local rollout")
print(effective_rank_entropy(student_local_total_rollout))
print("Baseline avg")
print(effective_rank_entropy(baseline_total_avg_attention))
print("Baseline avg")
print(effective_rank_entropy(baseline_total_rollout))
print("Bad student avg")
print(effective_rank_entropy(bad_student_total_avg_attention))
print("Bad student rollout")
print(effective_rank_entropy(bad_student_total_rollout))

Entropy based effective rank
Teacher avg
tensor(31.1072)
Student avg
tensor(336.6037)
Student local avg
tensor(362.2327)
Teacher rollout
tensor(1.0003)
Student rollout
tensor(2.4227)
Student local rollout
tensor(2.1826)
Baseline avg
tensor(238.3626)
Baseline avg
tensor(1.9391)
Bad student avg
tensor(370.2692)
Bad student rollout
tensor(2.5825)


# B. Layer-wise divergence profiles

In [ ]:
import os; os.makedirs("../images", exist_ok=True)
# Layer-wise divergence: each student vs teacher_avg_attentions (per layer)
STUDENT_KEYS = ["baseline", "bad_student", "student", "student_local"]
MODEL_COLORS = {
    "baseline":      "tab:blue",
    "bad_student":   "tab:red",
    "student":       "tab:green",
    "student_local": "tab:purple",
}

layer_idx = list(range(11))

cosine_by_model    = {}
frobenius_by_model = {}
js_by_model        = {}

for name in STUDENT_KEYS:
    s_attns = model_attentions[name]
    cos_v, frob_v, js_v = [], [], []
    for k in range(11):
        t = teacher_avg_attentions[k]
        s = s_attns[k]
        cos_v.append(cosine_similarity_tensor(t, s).item())
        frob_v.append(frobenius_difference(t, s).item())
        js_v.append(js_divergence_attention(t, s).item())
    cosine_by_model[name]    = cos_v
    frobenius_by_model[name] = frob_v
    js_by_model[name]        = js_v

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (data, title) in zip(axes, [
    (cosine_by_model,    "Cosine similarity"),
    (frobenius_by_model, "Frobenius distance"),
    (js_by_model,        "JS divergence"),
]):
    for name in STUDENT_KEYS:
        ax.plot(layer_idx, data[name], label=MODEL_LABELS[name],
                color=MODEL_COLORS[name], marker="o", linewidth=2, markersize=5)
    ax.set_xlabel("Layer index (student)")
    ax.set_title(title)
    ax.set_xticks(layer_idx)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle("Layer-wise divergence profiles (teacher_avg vs student)", y=1.02)
plt.tight_layout()
plt.savefig("../images/imdb_layer_divergence_profiles.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ../images/imdb_layer_divergence_profiles.png")


# C. Per-head similarity heatmaps

In [ ]:
# Compute per-head attention patterns (N=50, max_seq=128)
eval_prompts_50 = [build_prompt(t) for t in eval_texts[:50]]

head_patterns = {}
for name, model in MODELS.items():
    print(f"Head patterns for {MODEL_LABELS[name]}...")
    model.to(device)
    head_patterns[name] = compute_head_avg_patterns(
        model, eval_prompts_50, tokenizer, device, n=50, max_seq=128
    )
    model.cpu()
    torch.cuda.empty_cache()

print("Done. Shape:", head_patterns["teacher"].shape)


In [ ]:
import os; os.makedirs("../images", exist_ok=True)
# Per-head cosine-similarity heatmap: teacher vs each student
# 32x32 matrix averaged over 11 matched layer pairs
teacher_hp = head_patterns["teacher"]   # [22, n_heads, 128]
n_heads    = teacher_hp.shape[1]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name in zip(axes, STUDENT_KEYS):
    student_hp = head_patterns[name]    # [11, n_heads, 128]
    sim_sum = torch.zeros(n_heads, n_heads)
    for k in range(11):
        t_paired = (teacher_hp[2*k] + teacher_hp[2*k+1]) / 2
        s        = student_hp[k]
        t_norm   = t_paired / (t_paired.norm(dim=1, keepdim=True) + 1e-12)
        s_norm   = s        / (s.norm(dim=1, keepdim=True) + 1e-12)
        sim_sum += (t_norm @ s_norm.T)
    sim_avg = (sim_sum / 11).numpy()
    im = ax.imshow(sim_avg, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
    ax.set_title(f"Teacher vs {MODEL_LABELS[name]}")
    ax.set_xlabel("Student head")
    ax.set_ylabel("Teacher head")
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle("Per-head cosine similarity (avg over 11 matched layer pairs, N=50)", y=1.02)
plt.tight_layout()
plt.savefig("../images/imdb_head_similarity_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ../images/imdb_head_similarity_heatmap.png")


# D. Hidden states — Linear CKA

In [ ]:
# Compute hidden states for all models (N=100, max_seq=512)
eval_prompts_100 = [build_prompt(t) for t in eval_texts[:100]]

hidden_states_all = {}
for name, model in MODELS.items():
    print(f"Hidden states for {MODEL_LABELS[name]}...")
    model.to(device)
    hidden_states_all[name] = compute_hidden_states(
        model, eval_prompts_100, tokenizer, device, n=100, max_seq=512
    )
    model.cpu()
    torch.cuda.empty_cache()

print("Shapes:", {k: tuple(v.shape) for k, v in hidden_states_all.items()})


In [ ]:
import os; os.makedirs("../images", exist_ok=True)
# Linear CKA heatmap: teacher layers vs student layers (N=100)
import numpy as np

teacher_hs = hidden_states_all["teacher"]  # [23, 100, 2048]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, name in zip(axes, STUDENT_KEYS):
    student_hs = hidden_states_all[name]   # [12, 100, 2048]
    n_t = teacher_hs.shape[0]
    n_s = student_hs.shape[0]
    cka_mat = np.zeros((n_t, n_s))
    for i in range(n_t):
        for j in range(n_s):
            cka_mat[i, j] = linear_cka(teacher_hs[i], student_hs[j])
    im = ax.imshow(cka_mat, vmin=0, vmax=1, cmap="Blues", aspect="auto")
    ax.set_title(f"Teacher vs {MODEL_LABELS[name]}")
    ax.set_xlabel("Student layer")
    ax.set_ylabel("Teacher layer")
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle("Linear CKA: teacher vs student hidden states (N=100)", y=1.02)
plt.tight_layout()
plt.savefig("../images/imdb_cka_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ../images/imdb_cka_heatmap.png")
